<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/NMT_project_Darrick_Pang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset

In [ ]:
results = []
epochs = 10
batch_size = 256
training_samples = 1500000
model = "Transformer"
range_bleu = 200

source_vocab = 30000
target_vocab = 30000
embedding_dim = 256
latent_dim = 512

In [ ]:
# pip install evaluate

In [ ]:
# pip install sacrebleu

In [ ]:
from evaluate import load
bleu = load("sacrebleu")

In [ ]:
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]
val_data = dataset["validation"]

In [ ]:
print(train_data["translation"])
print(val_data)

In [ ]:
source_text = [german["de"] for german in train_data["translation"]]
# print(source_text)

target_text = [english["en"] for english in train_data["translation"]]
# print(target_text)

# Add start and end tokens to target text
target_text_with_tokens = ['<start> ' + text + ' <end>' for text in target_text]


source_tokenizer = Tokenizer(num_words=30000, filters='')
target_tokenizer = Tokenizer(num_words=30000, filters='')

source_tokenizer.fit_on_texts(source_text)
target_tokenizer.fit_on_texts(target_text_with_tokens)

source_sequence = source_tokenizer.texts_to_sequences(source_text)
target_sequence = target_tokenizer.texts_to_sequences(target_text_with_tokens)

max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')

In [ ]:
# encoder_inputs = layers.Input(shape=(max_src_len,))
# encoder_embeddings = layers.Embedding(source_vocab, embedding_dim)(encoder_inputs)
# encoder_lstm = layers.Bidirectional(layers.LSTM(latent_dim, return_sequences=True, return_state=True))
# encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embeddings)

# # Concatenate forward and backward states
# state_h = layers.Concatenate()([forward_h, backward_h])
# state_c = layers.Concatenate()([forward_c, backward_c])


# decoder_inputs = layers.Input(shape=(1,))
# decoder_embeddings = layers.Embedding(target_vocab, embedding_dim)(decoder_inputs)
# decoder_lstm = layers.LSTM(latent_dim * 2, return_sequences=True, return_state=True)
# decoder_outputs, _, _ = decoder_lstm(decoder_embeddings, initial_state=[state_h, state_c])

# attention = layers.Attention()([decoder_outputs, encoder_outputs])
# decoder_concat = layers.Concatenate(axis=-1)([decoder_outputs, attention])

# # Add a Dense layer for outputting probabilities for each word in the target vocabulary
# decoder_dense = layers.Dense(target_vocab, activation='softmax')
# decoder_outputs = decoder_dense(decoder_concat)


# model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ---- Hyperparams (small, fast) ----
d_model = 256          # model width
num_heads = 4
d_ff = 1024            # FFN hidden
num_enc = 3            # encoder layers
num_dec = 3            # decoder layers
dropout_rate = 0.1
v_src = source_vocab   # 30_000 from your code
v_tgt = target_vocab   # 30_000 from your code

# ---- Positional Encoding ----
class PositionalEncoding(layers.Layer):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        import numpy as np
        pe = np.zeros((max_len, d_model), dtype="float32")
        position = np.arange(0, max_len)[:, None]
        div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div)
        pe[:, 1::2] = np.cos(position * div)
        self.pe = tf.constant(pe[None, ...])   # [1, max_len, d_model]
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# ---- Masks ----
def padding_mask(x):
    # x: [B, T] int32
    return tf.cast(tf.equal(x, 0), tf.bool)  # True where PAD

def causal_mask(T):
    return tf.linalg.band_part(tf.ones((T, T), dtype=tf.bool), -1, 0)  # lower-triangular True

# ---- Encoder/Decoder Blocks ----
def encoder_block(x, pad_mask):
    # x: [B, T, d_model]
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = attn(query=x, value=x, key=x, attention_mask=~pad_mask[:, None, None, :])  # mask True=keep
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    return x

def decoder_block(x, enc_out, look_mask, enc_pad_mask):
    # 1️⃣ Self-attention
    self_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = self_attn(query=x, value=x, key=x, attention_mask=look_mask[:, None, :, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 2️⃣ Cross-attention (encoder–decoder)
    cross_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = cross_attn(query=x, value=enc_out, key=enc_out, attention_mask=~enc_pad_mask[:, None, None, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 3️⃣ Feed-forward
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))
    return x


# ---- Inputs (reuse your max_src_len / max_tgt_len) ----
enc_inp = layers.Input(shape=(max_src_len,), name="enc_tokens")
dec_inp = layers.Input(shape=(max_tgt_len,), name="dec_tokens")  # teacher-forced full sequence

# Embeddings (+ tie dims)
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)

# Add positional encodings
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)

# Masks
enc_pad = layers.Lambda(lambda x: tf.cast(tf.equal(x, 0), tf.bool), name="enc_pad")(enc_inp)

# Decoder look-ahead + padding mask
def make_lookahead_mask(x):
    seq_len = tf.shape(x)[1]
    mask = tf.cast(tf.not_equal(x, 0), tf.bool)
    mask = tf.logical_and(tf.tile(mask[:, None, :], [1, seq_len, 1]),
                          tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0))
    return mask

look = layers.Lambda(make_lookahead_mask, name="lookahead_mask")(dec_inp)

# Encoder stack
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

# Decoder stack
x = dec_x
for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)

# Output projection
logits = layers.Dense(v_tgt, activation="softmax")(x)

transformer = Model([enc_inp, dec_inp], logits)


In [ ]:
# Label smoothing (improves BLEU a bit)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    # label_smoothing=0.1 # Removed unsupported argument
)

from tensorflow.keras import backend as K

def smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1):
    y_true = tf.cast(y_true, tf.int32)
    num_classes = tf.shape(y_pred)[-1]
    y_true_one_hot = tf.one_hot(y_true, depth=num_classes)
    smooth_positives = 1.0 - label_smoothing
    smooth_negatives = label_smoothing / tf.cast(num_classes, tf.float32)
    y_true_smooth = y_true_one_hot * smooth_positives + smooth_negatives
    loss = -tf.reduce_sum(y_true_smooth * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(loss)

# Noam-style schedule (Transformer baseline)
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return (self.d_model ** -0.5) * tf.minimum(step ** -0.5, step * (self.warmup_steps ** -1.5))

lr = NoamSchedule(d_model, warmup_steps=4000)
opt = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

transformer.compile(optimizer=opt, loss=smoothed_sparse_categorical_crossentropy, metrics=["accuracy"])

In [ ]:
start_time = time.time()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

transformer.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr]
)

end_time = time.time()
elapsed = end_time - start_time
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

In [ ]:
# start_time = time.time()

# model.fit([encoder_input, decoder_input], np.expand_dims(decoder_target, -1), batch_size=batch_size, epochs=epochs, validation_split=0.1)

# end_time = time.time()
# elapsed = end_time - start_time
# print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

In [ ]:
# encoder_inference_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# decoder_state_input_h = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
# decoder_state_input_c = layers.Input(shape=(latent_dim * 2,)) # Corrected shape
# encoder_outputs_input = layers.Input(shape=(max_src_len, latent_dim*2)) # Input for encoder outputs, shape should match encoder_outputs


# decoder_output, state_h_inf, state_c_inf = decoder_lstm(
#     decoder_embeddings, initial_state=[decoder_state_input_h, decoder_state_input_c]
# )

# attention_inf = layers.Attention()([decoder_output, encoder_outputs_input])
# decoder_concat_inf = layers.Concatenate(axis=-1)([decoder_output, attention_inf])


# decoder_output_inf = decoder_dense(decoder_concat_inf)

# decoder_model = Model(
#     [decoder_inputs, decoder_state_input_h, decoder_state_input_c, encoder_outputs_input],
#     [decoder_output_inf, state_h_inf, state_c_inf]
# )

In [ ]:
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
    # ---- Encode the source sentence ----
    src_seq = source_tokenizer.texts_to_sequences([sentence])
    src_seq = pad_sequences(src_seq, maxlen=max_src_len, padding='post')

    start_id = target_tokenizer.word_index['<start>']
    end_id   = target_tokenizer.word_index['<end>']

    # Each beam is (sequence_so_far, cumulative_log_prob)
    beams = [([start_id], 0.0)]

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            # Stop expanding finished hypotheses
            if seq[-1] == end_id:
                new_beams.append((seq, score))
                continue

            dec_seq = pad_sequences([seq], maxlen=max_tgt_len, padding='post')
            preds = transformer.predict([src_seq, dec_seq], verbose=0)
            probs = preds[0, len(seq)-1, :]  # distribution for next token

            # pick top-k candidates
            top_ids = np.argsort(probs)[-beam_width:]
            for t in top_ids:
                new_seq = seq + [int(t)]
                new_score = score + np.log(probs[t] + 1e-9)
                new_beams.append((new_seq, new_score))

        # Keep the best `beam_width` beams
        # ---- Length normalization function ----
        def length_norm(score, length, alpha=alpha):
            return score / ((5 + length) / 6) ** alpha

        # Keep the best normalized beams
        beams = sorted(
            new_beams,
            key=lambda x: length_norm(x[1], len(x[0])),
            reverse=True
        )[:beam_width]

        # Early-stop if all beams ended
        if all(seq[-1] == end_id for seq, _ in beams):
            break

    # Take best-scoring beam
    best_seq = beams[0][0]
    words = [target_tokenizer.index_word.get(i, '') for i in best_seq[1:] if i not in (0, end_id)]
    return ' '.join(words)

In [ ]:
print(translate("Das ist ein Test."))

In [ ]:
# Generate predictions on a small subset of validation data
predictions = []
references = []

for i in range(range_bleu):  # 200 sentences for demo, increase later
    de_sentence = test_data[i]["translation"]["de"]
    en_reference = test_data[i]["translation"]["en"]

    en_predicted = translate(de_sentence, beam_width=6, alpha=0.7)

    predictions.append(en_predicted)
    references.append([en_reference])  # sacreBLEU expects list of list

result = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {result['score']:.2f}")

In [ ]:
results.append({"Epochs": epochs, "Batch Size": batch_size, "Training Time": elapsed, "BLEU": result['score'], "Training Data": training_samples, "Model": model, "Range BLEU": range_bleu})

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
df.to_excel('NMT_output_Darrick_Pang.xlsx', sheet_name='MyData')

In [ ]:
df = pd.read_excel('NMT_output_Darrick_Pang.xlsx')
df

In [ ]:
# !jupyter nbconvert --clear-output --to notebook --output="NMT_project_Darrick_Pang.ipynb" "NMT_project_Darrick_Pang_original.ipynb"

Initially, I stared off with using an LSTM. LSTM was a good starting point because it handles sequential data better than an RNN, works well for datasets using fewer than one million pairs, and can give a general idea of how translations work. The downside is that LSTM has a lower BLEU score. With LSTM and bidirectional LSTM, my BLEU score ranged between 2 and 5.

The issue with LSTM and bidirectional LSTM is that they are either not suited for full-scale WMT tasks or large-scale translations. That is where the transformer comes in. This tool is a better choice because we can score consistently higher on the BLEU metric than LSTM and bidirectional LSTM.

After seeing some of the limitations of LSTM, like being useful for smaller datasets, it seemed using a Transformer might be a better choice since it can consistently score higher on the BLEU metric than the LSTM.

After we load  and split the dataset,
```python
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]

```
we preprocess the data because we want to extract the German and English translations so we can tokenize each sentence and convert them to integer sequences. This step is needed because neural networks cannot process words, only numerical digits.

We then include this part
```python
max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')
```
because neural networks expect fixed-length inputs. Since some sentences are short, we can pad them with zeros to keep all sentences the same length. This gives deep learning models efficiency in batch processing.

Now we reach the main part of the code where we will build the Transformer-based neural machine translation. We start off with these hyperparemeters:
```python
d_model = 256
num_heads = 4
d_ff = 1024
num_enc = 3
num_dec = 3
dropout_rate = 0.1
v_src = source_vocab
v_tgt = target_vocab
```
.
These parameters determine the model complexity and its training speed. We have
```python
d_model
```
whose purpose is to give the dimentionality of the embeddings and capture meaning, context, and position in a sentence.

The purpose of
```python
num_heads
```
is to focus on different aspects of the sentence so the model can capture multiple relationsips. Adding
```python
d_ff = 1024
```
adds depth and abstraction after attention. This is a feedforward network that will process each token's vector to enrich representation.


The next ones are
```python
num_enc = 3
num_dec = 3
```
We need these because each encoder will refine sentence-level representation while each decoder will refine translation generation.
Then there's
```python
dropout_rate = 0.1
```
to prevent overfitting, and we have
```python
v_src, v_tgt
```
which are the vocabulary sizes. Each are set to 30,000. This will allow us to define the embeddings and define output dimensions.  

We add
```python
class PositionalEncoding(layers.Layer):
```
because we want our transformer know that the order matters. That is, "man eats food" is not the same as "food eats man". Transformers do not process words sequentially like an RNN. Then we add masks
```python
def padding_mask(x):
def causal_mask(T):
```
because we want to ignore padded zeros and ensure the decoder only see past tokens. Otherwise, the model may cheat and look ahead and generate new words. This part is needed for attention mechanisms.
Then we add the encoding and decoding blocks
```python
def encoder_block(x, pad_mask):
def decoder_block(x, enc_out, look_mask, enc_pad_mask):
```
to process source sentence embeddings and generate translated tokens one by one. We need both of them to output high-level contextual embeddings to represent the meaning of the source sentence and to enable the model to understand what is translated and what will be translated next.
This part
```python
enc_inp = layers.Input(shape=(max_src_len,))
dec_inp = layers.Input(shape=(max_tgt_len,))
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)
```
is used to define model tensors and convert token ID's into dense vectors. We need this because embedding layers will transformed the word ID into a learned word representation. This also will be needed for the positional encodings to ensure the model can understand word order relationships without sequential recurrence. The code for this part is
```python
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)
```
Now, we build an encoder and decoder stack
```python
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)
```
so we allow our Transformer model to better understand the contextual meaning of the sentences. Finally, we have an output projection
```python
logits = layers.Dense(v_tgt, activation="softmax")(x)
transformer = Model([enc_inp, dec_inp], logits)
```
This gives us a probability distribution for each time step of possible next words in our translation.

Overall, this step was needed to build the Transformer architecture to translate each sentence by having the model first understand that order is important. Then we use the encoder block to process the sentence and understand the context, and then use the decoder block to generate the translation using the output from the encoder.

In this step, we see the code
```python
def smoothed_sparse_categorical_crossentropy(...):
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
```
For crossentropy, we want to measure the model performance by measuring how well our model's predicted word matches the actual words so we can reduce loss and improve the translation quality and the BLEU score. For Noam scheduling, we want to increase the learning rate then decrease it because this would improve accuracy as well. Think of it as working out. If we stay on the heavy weights too long, it could hurt our performance in subsequent workout sessions. SImilarly, if we keep the learning rate too high, it would hurt accuracy as the model continues to train. To maximize performance, we want to lower the rate as time goes on.

After compiling and training the model, we generate translations one token at a time by beam searching to find the most likely translations with alpha providing length normalizations to prevent the short sentences from dominating.
```python
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
```
Then we evaluate the BLEU score.
```python
result = bleu.compute(predictions=predictions, references=references)
```